# 22.3 從零實作 DINO 核心

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/22-distillation/)

先執行下一格的環境格，再閱讀、執行最後的完整實驗。各節互相獨立，不需要先跑其他節。

最後一格存著這段程式在 CPU 上實際執行的輸出；執行環境與其他實驗紀錄見網站的〈驗證範圍〉。範例不用預訓練權重，也不需要下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.6.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
"""真正更新 tiny ViT，並從中間 checkpoint 恢復到與連續訓練相同的狀態。"""
import json
from pathlib import Path
import time

import torch

from miniyolo.figures import save_svg, use_svg_text
from miniyolo.self_distillation import DINOTrainer, extract_features, feature_diagnostics
from miniyolo.vision_data import ColorRectangles


def assert_same(left, right):
    if isinstance(left, torch.Tensor):
        assert torch.equal(left, right)
    elif isinstance(left, dict):
        assert left.keys() == right.keys()
        for key in left:
            assert_same(left[key], right[key])
    elif isinstance(left, (list, tuple)):
        assert len(left) == len(right)
        for a, b in zip(left, right):
            assert_same(a, b)
    else:
        assert left == right


def main(steps=160):
    if not 80 <= steps <= 200:
        raise ValueError("課堂 CPU 示範 steps 必須介於80與200")
    torch.set_num_threads(2)
    torch.manual_seed(7)
    output = Path("artifacts/runs/dino")
    output.mkdir(parents=True, exist_ok=True)
    images = ColorRectangles(samples=128, seed=101).images  # labels/bboxes 不進 SSL
    trainer = DINOTrainer(seed=7)
    before = {name: tensor.clone() for name, tensor in trainer.student.backbone.state_dict().items()}
    start = time.perf_counter()
    midpoint = steps // 2
    history = trainer.train_steps(images, midpoint)
    checkpoint = output / "22-distillation-midpoint.pt"
    trainer.save_checkpoint(checkpoint)
    next_views = trainer.peek_next_views(images)
    continuation = trainer.train_steps(images, steps - midpoint)
    continuous_next_views = trainer.peek_next_views(images)
    resumed = DINOTrainer.load_checkpoint(checkpoint)
    assert all(torch.equal(a, b) for a, b in zip(next_views, resumed.peek_next_views(images)))
    resume_history = resumed.train_steps(images, steps - midpoint)
    assert continuation == resume_history
    for component in ("student", "teacher", "optimizer", "loss"):
        assert_same(getattr(trainer, component).state_dict(), getattr(resumed, component).state_dict())
    assert trainer.step == resumed.step == steps
    assert torch.equal(trainer.generator.get_state(), resumed.generator.get_state())
    assert all(torch.equal(a, b) for a, b in zip(continuous_next_views, resumed.peek_next_views(images)))
    elapsed = time.perf_counter() - start
    history += continuation
    backbone_changed = any(not torch.equal(before[name], tensor)
                           for name, tensor in trainer.student.backbone.state_dict().items() if not name.startswith("head."))
    assert backbone_changed and all(parameter.grad is None for parameter in trainer.teacher.parameters())
    features = extract_features(trainer.teacher.backbone, images)
    with torch.no_grad():
        probabilities = trainer.loss.teacher_probabilities(trainer.teacher(images))
    result = {"seed": 7, "device": "cpu", "steps": steps, "checkpoint_step": midpoint,
              "train_samples": len(images), "batch_size": trainer.config.batch_size,
              "backbone": trainer.backbone_config, "training_config": vars(trainer.config),
              "first_step": history[0], "last_step": history[-1], "history": history,
              "backbone_weights_changed": backbone_changed, "teacher_has_no_grad": True,
              "resume": {"next_random_views_identical": True, "student_teacher_optimizer_center_identical": True,
                         "loss_history_identical": True, "final_step": resumed.step},
              "diagnostics_train": feature_diagnostics(features, probabilities),
              "elapsed_training_and_resume_seconds": elapsed,
              "checkpoint_path": str(checkpoint),
              "simplifications": "2 global views; ordinary MLP and no weight normalization; fixed temperatures/LR/EMA; no multi-crop or official schedules",
              "limitation": "DINO2021 mechanism demonstration from scratch; loss/diagnostics are not downstream or natural-image capability proof"}
    trainer.save_checkpoint(output / "22-distillation-final.pt")
    (output / "22-distillation.json").write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\n")
    use_svg_text()
    import matplotlib.pyplot as plt
    figure, axes = plt.subplots(1, 2, figsize=(10, 3.4))
    axes[0].plot([row["step"] for row in history], [row["loss"] for row in history])
    axes[0].set(xlabel="student update step", ylabel="cross-view CE", title="實際160步：loss 不代表特徵用途")
    axes[1].plot([row["step"] for row in history], [row["teacher_entropy"] for row in history])
    axes[1].set(xlabel="student update step", ylabel="teacher output entropy", title="centered teacher 分佈的平均熵")
    figure.tight_layout()
    save_svg(figure, output / "22-distillation.svg", "DINO 短訓練實測", "跨 view CE 與 teacher entropy 隨更新步數變化")
    plt.close(figure)
    print(json.dumps({key: value for key, value in result.items() if key != "history"}, ensure_ascii=False, indent=2))
    print(f"actual history: {output / '22-distillation.json'}")


if __name__ == "__main__":
    main()


{
  "seed": 7,
  "device": "cpu",
  "steps": 160,
  "checkpoint_step": 80,
  "train_samples": 128,
  "batch_size": 24,
  "backbone": {
    "image_size": 32,
    "patch_size": 8,
    "embed_dim": 32,
    "depth": 2,
    "num_heads": 4,
    "mlp_ratio": 2,
    "num_classes": 2,
    "dropout": 0.0
  },
  "training_config": {
    "output_dim": 16,
    "hidden_dim": 64,
    "bottleneck_dim": 16,
    "batch_size": 24,
    "learning_rate": 0.0005,
    "student_temperature": 0.15,
    "teacher_temperature": 0.08,
    "teacher_momentum": 0.95,
    "center_momentum": 0.9,
    "global_scale": [
      0.65,
      1.0
    ]
  },
  "first_step": {
    "step": 1,
    "loss": 1.9427120685577393,
    "teacher_entropy": 1.7583107948303223,
    "student_grad_norm": 24.119150161743164
  },
  "last_step": {
    "step": 160,
    "loss": 1.8742752075195312,
    "teacher_entropy": 1.8416004180908203,
    "student_grad_norm": 0.9461426734924316
  },
  "backbone_weights_changed": true,
  "teacher_has_no_grad": 